# Day 4

## Tokenizing with code

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Daniil Shevchenko and I love railways")

In [3]:
tokens

[12194,
 922,
 1308,
 382,
 94804,
 311,
 3627,
 85,
 3184,
 3593,
 326,
 357,
 3047,
 21665,
 2771]

In [4]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
94804 =  Dani
311 = il
3627 =  She
85 = v
3184 = chen
3593 = ko
326 =  and
357 =  I
3047 =  love
21665 =  rail
2771 = ways


In [5]:
encoding.decode([3047])

' love'

# And another topic!

### The Illusion of "memory"

In [6]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [7]:
from openai import OpenAI

openai = OpenAI()

### A message to OpenAI is a list of dicts

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer in Ukrainian"},
    {"role": "user", "content": "Привіт! Мене звати Даниїл Шевченко"}
    ]

In [9]:
response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=messages)
response.choices[0].message.content

'Привіт, Даниїле! Радий знайомству 😊 Чем можу допомогти?'

### OK let's now ask a follow-up question

In [10]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer in Ukrainian"},
    {"role": "user", "content": "Як мене звати?"}
    ]

In [11]:
response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=messages)
response.choices[0].message.content

'Я не знаю — ви ще не називали свого імені.'

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer in Ukrainian"},
    {"role": "user", "content": "Привіт! Мене звати Даниїл Шевченко"},
    {"role": "assistant", "content": "Привіт, Даниїле! Радий знайомству 😊 Чем можу допомогти?"},
    {"role": "user", "content": "Як мене звати?"}
    ]

In [13]:
response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=messages)
response.choices[0].message.content

'Тебе звати Даниїл Шевченко.'

## To recap

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

